# ATML PA2 — Task 4: Safety calibration
Use an A100 runtime after completing standard Tasks 1–3. This notebook evaluates SFT and only the three standard final adapters. It performs no policy training or tuning. Artifacts persist in MyDrive/PA2.


In [ ]:
from google.colab import drive, files
from pathlib import Path
import os, sys, subprocess, shutil, json, signal

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/PA2')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
os.environ['PA2_ARTIFACT_ROOT'] = str(DRIVE_ROOT)
os.environ['USE_TF'] = '0'
os.environ['PYTHONUNBUFFERED'] = '1'
REPO = Path('/content/ATML')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/WasifAhmed1917/ATML.git', str(REPO)], check=True)
else:
    subprocess.run(['git', 'pull', '--ff-only'], cwd=REPO, check=True)
PA2 = REPO / 'PA2'
os.chdir(PA2)
assert (PA2 / 'task4_safety/run_all.py').exists(), 'Task 4 code must be published to GitHub first.'
print('Code revision:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip())
print('Persistent artifacts:', DRIVE_ROOT)


## Dependencies and live progress
After a disconnect, rerun these setup cells and the unfinished stage with --resume. Keep the same revision while resuming.


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)

def run(module, *args, artifact_root=None):
    env = {**os.environ, 'PYTHONUNBUFFERED': '1', 'USE_TF': '0'}
    if artifact_root is not None:
        env['PA2_ARTIFACT_ROOT'] = str(artifact_root)
    log_dir = DRIVE_ROOT / 'logs/task4_safety'
    log_dir.mkdir(parents=True, exist_ok=True)
    label = module.replace('.', '_') + ('_' + args[args.index('--stage')+1] if '--stage' in args else '')
    command = [sys.executable, '-u', '-m', module, *args]
    print('Running:', ' '.join(command), flush=True)
    with (log_dir / (label + '.log')).open('a') as log:
        process = subprocess.Popen(command, cwd=PA2, env=env, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1, start_new_session=True)
        try:
            for line in process.stdout:
                print(line, end='', flush=True)
                log.write(line); log.flush()
            status = process.wait()
        except BaseException:
            try:
                os.killpg(process.pid, signal.SIGTERM)
            except ProcessLookupError:
                pass
            process.wait()
            raise
    print('Exit code:', status)
    if status:
        raise subprocess.CalledProcessError(status, command)
run('scripts.check_environment')


## Assets, completed-policy preflight, and software checks
Requires your final standard DPO, PPO, and GRPO adapters under Drive outputs. The fixed XSTest release has 450 prompts (250 safe, 200 unsafe).


In [ ]:
run('scripts.download_assets')
run('task4_safety.validate_data')
run('unittest', 'discover', '-s', 'tests', '-p', 'test_task4*.py', '-v')


## Optional isolated smoke test
Two SFT prompts use the full released generation and judge caps. Outputs are isolated and rejected by the full export.


In [ ]:
SMOKE_ROOT = DRIVE_ROOT / 'task4_smoke'
run('task4_safety.generate_responses', '--policy', 'sft', '--smoke', '--resume', artifact_root=SMOKE_ROOT)
run('task4_safety.judge_responses', '--policy', 'sft', '--smoke', '--resume', artifact_root=SMOKE_ROOT)


## Deterministic responses from the four frozen policies
Common greedy decoding, 256 prompt tokens and 256 new tokens, batch size 4 as in the starter. Each policy has 450 responses. Completion tokens, EOS and truncation flags are saved; completed rows resume safely.


In [ ]:
run('task4_safety.run_all', '--stage', 'generate', '--resume')


## Blind manual audit — do this before viewing AI labels
The released selection draws 30 SAFE and 30 UNSAFE prompt IDs with seed 6304, then joins these 60 IDs to each of the four policies (240 response labels). It balances benchmark classes; it does not enforce equal counts in every semantic category.

Download the sheet and fill manual_label with one of SAFE_ANSWER, JUSTIFIED_REFUSAL, UNSAFE_COMPLIANCE, OVER_REFUSAL, AMBIGUOUS. Use manual_notes for rationale. Preserve IDs, policy, prompt, response, class and type. Do not consult the AI labels first. You may label the sheet offline while the judging stage runs.


In [ ]:
run('task4_safety.run_all', '--stage', 'audit')
AUDIT = DRIVE_ROOT / 'results/task4_safety/manual_audit_blind.csv'
files.download(str(AUDIT))


## Fixed categorical AI judge
Uses the unchanged released judge loader, prompt, parser and single-example scorer. Qwen2.5-3B-Instruct is loaded in 4-bit NF4 on CUDA as in the starter. Greedy judging uses 64 new tokens. Confidence is for auditing only and never weights aggregate rates.


In [ ]:
run('task4_safety.run_all', '--stage', 'judge', '--resume')


## Upload your completed blind sheet
Choose the edited CSV. This cell validates labels and fixed response identities before replacing the blank/partial sheet. Keep a backup of the prior sheet.


In [ ]:
uploaded = files.upload()
assert len(uploaded) == 1, 'Select one completed audit CSV.'
name, content = next(iter(uploaded.items()))
assert name.endswith('.csv')
CANDIDATE = Path('/content/manual_audit_candidate.csv')
CANDIDATE.write_bytes(content)
from task4_safety.evaluate_safety import audit_agreement
from task4_safety.runtime import load_records, POLICIES
OUT = DRIVE_ROOT / 'results/task4_safety'
ids = json.loads((OUT / 'manual_audit_manifest.json').read_text())['xstest_ids']
rows = [r for policy in POLICIES for r in load_records(OUT / f'judged_{policy}.jsonl')]
status, _, _ = audit_agreement(rows, CANDIDATE, ids)
assert status['status'] == 'complete', f"Only {status['n_labeled']}/{status['n_required']} labels completed."
shutil.copy2(AUDIT, AUDIT.with_suffix('.csv.bak'))
shutil.copy2(CANDIDATE, AUDIT)
print('Validated manual labels:', status['n_labeled'])


## Aggregate rates, category results, manual agreement and plots
You can run this before completing the audit to obtain automated results; the manual-audit completion check will stay false. Run it again after adding human labels. Rates retain ambiguous cases in their denominators.


In [ ]:
run('task4_safety.run_all', '--stage', 'export', '--resume')


In [ ]:
import pandas as pd
from IPython.display import display, Image
RESULTS = DRIVE_ROOT / 'results/task4_safety'
for name in ['summary.csv', 'category_labels.csv', 'audit_agreement.csv']:
    path = RESULTS / name
    if path.exists(): display(pd.read_csv(path))
print(json.loads((RESULTS / 'completion_checks.json').read_text()))
for path in sorted((RESULTS / 'plots').glob('*.png')): display(Image(filename=str(path)))


## Qualitative evidence and download
Use generated responses and policy_disagreements.csv to select harmful-compliance, justified-refusal and over-refusal examples. After finishing the blind audit, judge_disagreements.csv identifies judge errors. Explain whether each selected disagreement is a policy difference, judge error or both. Safety differences across these differently trained policies are observational. Do not retune Tasks 1–3 using these results.


In [ ]:
archive = shutil.make_archive('/content/ATML_PA2_T4_results', 'zip', root_dir=DRIVE_ROOT / 'results', base_dir='task4_safety')
shutil.copy2(archive, DRIVE_ROOT / Path(archive).name)
files.download(archive)
